# notebook_sync.ipynb — Kronos forecast experimentation

A **thin** notebook. It imports `forecast_engine` and calls the SAME functions the
dashboard and MCP server use. There is **zero duplicated model logic** here — that is
the "sync": notebook and dashboard literally share `forecast_engine.py`.

Honesty reminders (see SCOPE.md §0):
- The forecast is a **distribution** of sample paths, not a point.
- The model sees **price candles only** — no news / macro events.
- No fabricated confidence score anywhere.

## 1. Setup — add Kronos repo to path & import the engine

In [ ]:
import os, sys
# Make the project importable if running from elsewhere.
PROJECT_DIR = os.path.dirname(os.path.abspath('notebook_sync.ipynb')) if '__file__' not in dir() else os.path.dirname(__file__)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Optional: point at a non-default Kronos clone via env before importing.
# os.environ['KRONOS_DIR'] = r'C:\\path\\to\\Kronos'

import importlib
import forecast_engine as fe
import instruments
importlib.reload(fe)

print('Kronos dir:', fe.KRONOS_DIR)
print('Device   :', fe.DEFAULT_DEVICE)
print('Instruments:', [i.key for i in instruments.all_instruments()])

## 2. Pull + inspect raw data for one instrument

In [ ]:
INSTRUMENT = 'gold'  # silver | gold | wti | gbpjpy

df = fe.fetch_ohlcv(INSTRUMENT, lookback_days=500)
print('ticker used     :', df.attrs['ticker_used'])
print('volume reliable :', df.attrs['volume_is_reliable'])
print('rows            :', len(df))
df.tail()

## 3. Run a forecast and plot the fan inline

The shaded band (p10–p90) is the point of the chart — it shows what the model does
*not* know. The dashed median line is **not** "the answer".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Loads the model once (downloads weights on first run) and reuses it.
predictor = fe.load_predictor(device=fe.DEFAULT_DEVICE)

res = fe.run_forecast(INSTRUMENT, lookback=400, pred_len=20, n_samples=30, predictor=predictor)
print(res.summary_text())

In [ ]:
def plot_fan(res):
    hist = res.history
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(hist['timestamps'], hist['close'], color='black', lw=1.6, label='history')
    fx = res.future_timestamps
    # faint sample paths
    for i in range(res.sample_matrix.shape[0]):
        ax.plot(fx, res.sample_matrix[i], color='black', alpha=0.05, lw=0.8)
    # p10-p90 band (most prominent)
    ax.fill_between(fx, res.p10, res.p90, color='black', alpha=0.12, label='p10-p90 band')
    ax.fill_between(fx, res.p25, res.p75, color='black', alpha=0.20, label='p25-p75 band')
    ax.plot(fx, res.median, color='black', lw=1.6, ls='--', label='median (p50)')
    ax.axhline(res.last_close, color='gray', lw=1, ls=':')
    ax.set_title(f'{res.display_name} — forecast distribution (NOT a single prediction)')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.2)
    plt.tight_layout(); plt.show()

plot_fan(res)

In [ ]:
# Distribution table + skew
display(res.horizon_table().round(4))
display(res.skew_metrics.round(4))

## 4. Scratch cell — tweak params (T, top_p, lookback) and re-plot

Higher `T` / lower `top_p` => wider, more diverse paths. This changes the band
width — a reminder that the band is a modelling choice, not ground truth.

In [ ]:
res2 = fe.run_forecast(
    INSTRUMENT,
    lookback=400,
    pred_len=20,
    n_samples=40,
    T=1.1,
    top_p=0.9,
    predictor=predictor,
)
plot_fan(res2)
print(res2.skew_sentence())

## 5. (Optional) log this forecast & check accuracy over time

In [ ]:
import storage
run_id = storage.log_forecast(res)
print('logged run_id:', run_id)
n = storage.update_actuals(INSTRUMENT)
print('reconciled points:', n)
storage.accuracy_summary(INSTRUMENT)